### A5.4.7. Build Operator Fusion Pass

**Problem:**

Build a graph transformation pass that identifies sequences of element-wise operations and fuses them into a single compound operation, reducing memory traffic.

**Explanation:**

Operator fusion combines multiple operations into one so that intermediate results stay in registers instead of being written to and read from memory. The pass scans the graph for chains of fusible operations and replaces them with a single fused node.

How to build it:

1. Mark each operation as fusible or not (element-wise ops like add, relu, multiply are fusible).
2. Walk the graph. When you find a fusible node whose single consumer is also fusible, merge them: create a fused node that combines both operations, taking the first node's inputs and the second node's consumers.
3. Remove the original nodes and replace references with the fused node.
4. Repeat until no more fusions are possible.

**Example:**

A chain `add → relu → mul` fuses into a single `fused(add_relu_mul)` node. One kernel launch instead of three, and no intermediate memory allocation.

In [ ]:
FUSIBLE_OPS = {"add", "relu", "mul", "neg", "sigmoid"}


class Node:
    def __init__(self, name, op_type, inputs=None):
        self.name = name
        self.op_type = op_type
        self.inputs = inputs or []

    def __repr__(self):
        input_names = [node.name for node in self.inputs]
        return f"{self.name}({self.op_type}, inputs={input_names})"


def find_consumers(nodes, target):
    return [
        node for node in nodes
        if target in node.inputs
    ]


def fuse_operators(nodes):
    changed = True
    while changed:
        changed = False
        for node in list(nodes):
            if node.op_type not in FUSIBLE_OPS:
                continue
            consumers = find_consumers(nodes, node)
            if len(consumers) != 1:
                continue
            consumer = consumers[0]
            if consumer.op_type not in FUSIBLE_OPS:
                continue

            fused_name = f"{node.name}_{consumer.name}"
            fused_op = f"fused({node.op_type}+{consumer.op_type})"
            fused = Node(fused_name, fused_op, node.inputs)

            for other_node in nodes:
                other_node.inputs = [
                    fused if inp is consumer else inp
                    for inp in other_node.inputs
                ]

            nodes.remove(node)
            nodes.remove(consumer)
            nodes.append(fused)
            changed = True
            break

    return nodes


x = Node("x", "input")
w = Node("w", "input")
matmul = Node("matmul", "matmul", [x, w])
add_bias = Node("add_bias", "add", [matmul])
relu = Node("relu", "relu", [add_bias])
scale = Node("scale", "mul", [relu])

nodes = [x, w, matmul, add_bias, relu, scale]

print("Before fusion:")
for node in nodes:
    print(f"  {node}")

fused_nodes = fuse_operators(nodes)

print("\nAfter fusion:")
for node in fused_nodes:
    print(f"  {node}")

**References:**

📘 Chen, T. et al. (2018). *TVM: An Automated End-to-End Optimizing Compiler for Deep Learning* — Operator Fusion

---

[⬅️ Previous: Build Automatic Differentiation Engine](./06_build_automatic_differentiation_engine.ipynb) | [Next: Build Constant Folding Pass ➡️](./08_build_constant_folding_pass.ipynb)